# Lección 1 — Mecánica Lagrangiana con SymPy: Euler–Lagrange y coordenadas generalizadas
### Curso/Taller de Mecánica Clásica Computacional

> "La elección de coordenadas correctas convierte problemas difíciles en problemas naturales."

Este cuaderno es **100% autocontenido**, en español, y está diseñado como una clase extensa para modelar sistemas de mecánica clásica con el formalismo lagrangiano y resolver ecuaciones con **SymPy**.

## Tabla de contenidos

- [0. Imports y utilidades](#0.-Imports-y-utilidades)
- [1. Introducción y mapa del método](#1.-Introducción-y-mapa-del-método)
- [2. Recordatorio matemático mínimo](#2.-Recordatorio-matemático-mínimo)
- [3. Plantilla general para modelar sistemas](#3.-Plantilla-general-para-modelar-sistemas)
- [4. Problemas 1 y 2](#4.-Problemas-1-y-2)
  - [4.1 Péndulo simple](#4.1-Péndulo-simple)
  - [4.2 Oscilador masa–resorte 1D](#4.2-Oscilador-masa–resorte-1D)
- [5. Problemas 3 y 4](#5.-Problemas-3-y-4)
  - [5.1 Máquina de Atwood](#5.1-Máquina-de-Atwood)
  - [5.2 Partícula en potencial central](#5.2-Partícula-en-potencial-central)
- [6. Problemas 5 y 6](#6.-Problemas-5-y-6)
  - [6.1 Péndulo doble](#6.1-Péndulo-doble)
  - [6.2 Rodadura sin deslizamiento](#6.2-Rodadura-sin-deslizamiento)
- [7. Problemas 7 y 8](#7.-Problemas-7-y-8)
  - [7.1 Bead on a rotating hoop](#7.1-Bead-on-a-rotating-hoop)
  - [7.2 Disipación con función de Rayleigh](#7.2-Disipación-con-función-de-Rayleigh)
- [8. Comparativa: Manual vs `sympy.physics.mechanics`](#8.-Comparativa:-Manual-vs-sympy.physics.mechanics)
- [9. Buenas prácticas y errores comunes](#9.-Buenas-prácticas-y-errores-comunes)
- [10. Ejercicios para estudiantes](#10.-Ejercicios-para-estudiantes)
- [11. Referencias](#11.-Referencias)

### Cómo ejecutar / Reproducibilidad

- Entorno recomendado: Python 3.10+.
- Dependencia obligatoria: `sympy`.
- Dependencias opcionales: `numpy`, `matplotlib` (para gráficas y evaluaciones numéricas sencillas).
- Este notebook **no depende de internet** ni de archivos externos.

In [ ]:
# Instalación opcional (descomentar si hace falta en un entorno limpio)
# !pip install sympy numpy matplotlib

## 0. Imports y utilidades

En esta sección inicializamos SymPy con impresión matemática y creamos utilidades para:
1. Construir ecuaciones de Euler–Lagrange de forma manual.
2. Aplicar simplificaciones algebraicas/trigonométricas.
3. Construir ecuaciones con disipación de Rayleigh.

In [ ]:
import sympy as sp
from sympy import sin, cos
from sympy.physics.mechanics import dynamicsymbols, LagrangesMethod, ReferenceFrame, Point
import numpy as np
import matplotlib.pyplot as plt

sp.init_printing()
t = sp.symbols('t', real=True)
t_mechanics = sp.Symbol('t')  # dynamicsymbols está construido sobre Symbol('t') sin supuestos; 't' (real=True) se reserva para el enfoque manual con Function(t)

In [ ]:
def simplificar(expr):
    """Simplificación robusta para expresiones simbólicas."""
    return sp.trigsimp(sp.simplify(sp.factor(expr)))


def euler_lagrange_manual(L, qs, t, Qs=None, simpl=True):
    """
    Calcula ecuaciones de Euler–Lagrange para una lista de coordenadas q_i(t).
    Convención: d/dt(dL/dqdot) - dL/dq - Q_i = 0.
    """
    if Qs is None:
        Qs = [0] * len(qs)
    eqs = []
    for q, Q in zip(qs, Qs):
        qd = sp.diff(q, t)
        expr = sp.diff(sp.diff(L, qd), t) - sp.diff(L, q) - Q
        if simpl:
            expr = simplificar(expr)
        eqs.append(sp.Eq(expr, 0))
    return eqs


def euler_lagrange_con_rayleigh(L, R, qs, t, simpl=True):
    """
    Ecuación: d/dt(dL/dqdot) - dL/dq + dR/dqdot = 0.
    """
    eqs = []
    for q in qs:
        qd = sp.diff(q, t)
        expr = sp.diff(sp.diff(L, qd), t) - sp.diff(L, q) + sp.diff(R, qd)
        if simpl:
            expr = simplificar(expr)
        eqs.append(sp.Eq(expr, 0))
    return eqs


def energia_mecanica(T, V):
    return simplificar(T + V)

## 1. Introducción y mapa del método

El enfoque lagrangiano reemplaza la suma vectorial de fuerzas por un problema escalar:
\[
L(q_i,\dot q_i,t)=T-V
\]

y ecuaciones de evolución:
\[
\frac{d}{dt}\left(\frac{\partial L}{\partial \dot q_i}\right)-\frac{\partial L}{\partial q_i}=Q_i.
\]

### ¿Qué ganamos?
- Modelado natural con **coordenadas generalizadas**.
- Manejo elegante de restricciones.
- Identificación directa de cantidades conservadas (coordenadas cíclicas).

### Mapa práctico
1. Elegir coordenadas mínimas.
2. Escribir cinemática \(\mathbf{r}(q_i,t)\).
3. Construir \(T\) y \(V\).
4. Derivar E-L.
5. Simplificar, interpretar y estudiar casos límite.

## 2. Recordatorio matemático mínimo

- \(q_i(t)\): coordenadas generalizadas (funciones del tiempo).
- \(\dot q_i=\frac{dq_i}{dt}\), \(\ddot q_i=\frac{d^2q_i}{dt^2}\).
- En SymPy:
  - `sp.Function('q')(t)` para funciones del tiempo.
  - `dynamicsymbols('q')` en `sympy.physics.mechanics`.

La regla clave es distinguir entre **símbolo** (constante) y **función** (depende de \(t\)).

In [ ]:
# Diferencia entre símbolo y función del tiempo
q_simbolo = sp.symbols('q', real=True)
q_funcion = sp.Function('q')(t)

print('d/dt de símbolo q:', sp.diff(q_simbolo, t))
print('d/dt de función q(t):', sp.diff(q_funcion, t))

q_dyn = dynamicsymbols('q')
print('d/dt de dynamicsymbols q:', sp.diff(q_dyn, t_mechanics))

## 3. Plantilla general para modelar sistemas

**Checklist canónico**
1. Elegir \(q_i\).
2. Escribir \(\mathbf{r}(q_i,t)\).
3. Derivar \(\dot{\mathbf{r}}\).
4. Construir \(T\) y \(V\).
5. Formar \(L=T-V\).
6. Aplicar Euler–Lagrange.
7. Simplificar y analizar límites.

Mini-demostración con una partícula 1D en potencial \(V(x)\).

In [ ]:
x = sp.Function('x')(t)
m = sp.symbols('m', positive=True, real=True)
Vx = sp.Function('V')(x)

T_demo = sp.Rational(1,2)*m*sp.diff(x,t)**2
L_demo = T_demo - Vx
eq_demo = euler_lagrange_manual(L_demo, [x], t)[0]

print(sp.Eq(sp.Symbol('L'), L_demo))
print(eq_demo)
print("Interpretación: m*x_ddot + dV/dx = 0")

## 4. Problemas 1 y 2

---

### 4.1 Péndulo simple

**A) Planteamiento físico**
Masa puntual \(m\) unida a una varilla rígida de longitud \(\ell\), sin fricción.

**B) Coordenada generalizada**
\(\theta(t)\), ángulo respecto de la vertical. Una sola coordenada captura la restricción \(r=\ell\).

In [ ]:
# C) Cinemática, D) energías, E) Lagrangiano
m, l, g = sp.symbols('m l g', positive=True, real=True)
theta = sp.Function('theta')(t)

theta_dot = sp.diff(theta, t)
T_p = sp.Rational(1,2)*m*l**2*theta_dot**2
V_p = m*g*l*(1-sp.cos(theta))
L_p = T_p - V_p

print(sp.Eq(sp.Symbol('T'), T_p))
print(sp.Eq(sp.Symbol('V'), V_p))
print(sp.Eq(sp.Symbol('L'), L_p))

In [ ]:
# F) Euler–Lagrange manual
eq_p_manual = euler_lagrange_manual(L_p, [theta], t)[0]
print(eq_p_manual)

# Forma esperada: theta'' + (g/l) sin(theta)=0
eq_p_div = sp.simplify(eq_p_manual.lhs/(m*l**2))
print(sp.Eq(eq_p_div, 0))

In [ ]:
# G) Enfoque con sympy.physics.mechanics (LagrangesMethod)
th = dynamicsymbols('th')
L_p_mech = sp.Rational(1,2)*m*l**2*th.diff(t_mechanics)**2 - m*g*l*(1-sp.cos(th))
LM_p = LagrangesMethod(L_p_mech, [th])
eq_p_mech = sp.simplify(LM_p.form_lagranges_equations()[0])
print(sp.Eq(eq_p_mech, 0))

In [ ]:
# Uso de ReferenceFrame/Point para mostrar cinemática explícita
N = ReferenceFrame('N')
O = Point('O')
P = Point('P')
P.set_pos(O, l*sp.sin(th)*N.x - l*sp.cos(th)*N.y)
P.set_vel(N, P.pos_from(O).dt(N))

v2 = sp.simplify(P.vel(N).dot(P.vel(N)))
print(sp.Eq(sp.Symbol('v^2'), v2))

In [ ]:
# H) Conservación de energía + I) caso pequeño y gráfica de potencial
E_p = energia_mecanica(T_p, V_p)
print(sp.Eq(sp.Symbol('E'), E_p))

peq = sp.Eq(sp.diff(theta, t, 2) + (g/l)*theta, 0)
print(peq)
print('Frecuencia angular de oscilaciones pequeñas: omega0 = sqrt(g/l)')

# Gráfica del potencial (ejemplo numérico)
th_vals = np.linspace(-np.pi, np.pi, 400)
V_vals = (1 - np.cos(th_vals))
plt.figure(figsize=(6,3))
plt.plot(th_vals, V_vals)
plt.xlabel(r'$\theta$')
plt.ylabel(r'$V/(m g l)$')
plt.title('Péndulo simple: potencial adimensional')
plt.grid(True, alpha=0.3)
plt.show()

### 4.2 Oscilador masa–resorte 1D

**A)** Masa \(m\), resorte \(k\), sin amortiguamiento.  
**B)** Coordenada natural: elongación \(x(t)\).

In [ ]:
# C,D,E
m, k = sp.symbols('m k', positive=True, real=True)
x = sp.Function('x')(t)

T_o = sp.Rational(1,2)*m*sp.diff(x,t)**2
V_o = sp.Rational(1,2)*k*x**2
L_o = T_o - V_o

print(sp.Eq(sp.Symbol('L'), L_o))

In [ ]:
# F) Manual
eq_o = euler_lagrange_manual(L_o, [x], t)[0]
print(eq_o)
print(sp.Eq(sp.simplify(eq_o.lhs/m), 0))

In [ ]:
# G) mechanics
x_m = dynamicsymbols('x_m')
L_o_m = sp.Rational(1,2)*m*x_m.diff(t_mechanics)**2 - sp.Rational(1,2)*k*x_m**2
LM_o = LagrangesMethod(L_o_m, [x_m])
eq_o_m = sp.simplify(LM_o.form_lagranges_equations()[0])
print(sp.Eq(eq_o_m, 0))

In [ ]:
# H) Discusión e I) solución cerrada
sol_o = sp.dsolve(sp.Eq(sp.diff(x,t,2) + (k/m)*x, 0))
print(sol_o)
print("Comparación Newton: m*x_ddot = -k*x, coincide exactamente.")

## 5. Problemas 3 y 4

---

### 5.1 Máquina de Atwood

**A)** Dos masas \(m_1\) y \(m_2\) unidas por cuerda ideal sobre polea ideal.

**B)** Una coordenada \(x(t)\): desplazamiento de \(m_1\) hacia abajo.

In [ ]:
# C,D,E
m1, m2, g = sp.symbols('m1 m2 g', positive=True, real=True)
x = sp.Function('x')(t)
xd = sp.diff(x,t)

T_a = sp.Rational(1,2)*(m1+m2)*xd**2
V_a = -m1*g*x + m2*g*x  # según convención de signo elegida
L_a = T_a - V_a

print(sp.Eq(sp.Symbol('L'), sp.simplify(L_a)))

In [ ]:
# F) Manual
eq_a = euler_lagrange_manual(L_a, [x], t)[0]
print(eq_a)
acc_a = sp.solve(sp.Eq(eq_a.lhs,0), sp.diff(x,t,2))[0]
print(sp.Eq(sp.diff(x,t,2), sp.simplify(acc_a)))

In [ ]:
# G) mechanics
x_at = dynamicsymbols('x_at')
L_a_m = sp.Rational(1,2)*(m1+m2)*x_at.diff(t_mechanics)**2 - (m2-m1)*g*x_at
LM_a = LagrangesMethod(L_a_m, [x_at])
eq_a_m = sp.simplify(LM_a.form_lagranges_equations()[0])
print(sp.Eq(eq_a_m, 0))

In [ ]:
# H, I) Caso especial
acc_eq = sp.simplify((m1-m2)*g/(m1+m2))
print(sp.Eq(sp.Symbol('a'), acc_eq))
print('Caso especial m1 = m2 -> a = 0, como se espera por simetría.')

### 5.2 Partícula en coordenadas polares con potencial central \(V(r)\)

**A)** Movimiento plano en un potencial central.  
**B)** Coordenadas: \(q_1=r(t), q_2=\phi(t)\).

In [ ]:
# C,D,E
m = sp.symbols('m', positive=True, real=True)
r = sp.Function('r')(t)
phi = sp.Function('phi')(t)
V = sp.Function('V')

rd = sp.diff(r,t)
phid = sp.diff(phi,t)

T_c = sp.Rational(1,2)*m*(rd**2 + r**2*phid**2)
V_c = V(r)
L_c = T_c - V_c

print(sp.Eq(sp.Symbol('L'), L_c))

In [ ]:
# F) Manual
eq_r, eq_phi = euler_lagrange_manual(L_c, [r, phi], t, simpl=False)
print(eq_r)
print(eq_phi)

# Momento conjugado p_phi
p_phi_expr = sp.simplify(sp.diff(L_c, phid))
print(sp.Eq(sp.Symbol('p_phi'), p_phi_expr))
print('Como L no depende explícitamente de phi, p_phi es constante (coordenada cíclica).')

In [ ]:
# G) mechanics
rr, pp = dynamicsymbols('rr pp')
L_c_m = sp.Rational(1,2)*m*(rr.diff(t_mechanics)**2 + rr**2*pp.diff(t_mechanics)**2) - V(rr)
LM_c = LagrangesMethod(L_c_m, [rr, pp])
eqs_c_m = LM_c.form_lagranges_equations()
for i, eq in enumerate(eqs_c_m, start=1):
    print(sp.Eq(sp.simplify(eq), 0))

In [ ]:
# H) Energía y potencial efectivo, I) caso especial orbital
E_c = energia_mecanica(T_c, V_c)
Lz = sp.Symbol('Lz', real=True)
U_eff = V(r) + Lz**2/(2*m*r**2)
print(sp.Eq(sp.Symbol('E'), E_c))
print(sp.Eq(sp.Symbol('U_eff'), U_eff))
print('Discusión: el movimiento radial se interpreta como 1D en U_eff(r).')

## 6. Problemas 5 y 6

---

### 6.1 Péndulo doble

**A)** Dos masas puntuales \(m_1,m_2\) y longitudes \(\ell_1,\ell_2\).

**B)** Coordenadas generalizadas: \(\theta_1(t),\theta_2(t)\).

In [ ]:
# C,D,E
m1, m2, l1, l2, g = sp.symbols('m1 m2 l1 l2 g', positive=True, real=True)
th1 = sp.Function('th1')(t)
th2 = sp.Function('th2')(t)

x1 = l1*sp.sin(th1)
y1 = -l1*sp.cos(th1)
x2 = x1 + l2*sp.sin(th2)
y2 = y1 - l2*sp.cos(th2)

v1sq = sp.diff(x1,t)**2 + sp.diff(y1,t)**2
v2sq = sp.diff(x2,t)**2 + sp.diff(y2,t)**2

T_dp = sp.Rational(1,2)*m1*v1sq + sp.Rational(1,2)*m2*v2sq
V_dp = m1*g*y1 + m2*g*y2
L_dp = sp.simplify(T_dp - V_dp)

print(sp.Eq(sp.Symbol('L'), L_dp))

In [ ]:
# F) Manual (sin simplificación agresiva para evitar costo simbólico excesivo)
eq1_dp = sp.Eq(sp.diff(sp.diff(L_dp, sp.diff(th1,t)), t) - sp.diff(L_dp, th1), 0)
eq2_dp = sp.Eq(sp.diff(sp.diff(L_dp, sp.diff(th2,t)), t) - sp.diff(L_dp, th2), 0)
print(eq1_dp)
print(eq2_dp)

In [ ]:
# G) mechanics
q1m, q2m = dynamicsymbols('q1m q2m')
x1m = l1*sp.sin(q1m)
y1m = -l1*sp.cos(q1m)
x2m = x1m + l2*sp.sin(q2m)
y2m = y1m - l2*sp.cos(q2m)

v1m2 = sp.diff(x1m, t_mechanics)**2 + sp.diff(y1m, t_mechanics)**2
v2m2 = sp.diff(x2m, t_mechanics)**2 + sp.diff(y2m, t_mechanics)**2
L_dp_m = sp.Rational(1,2)*m1*v1m2 + sp.Rational(1,2)*m2*v2m2 - (m1*g*y1m + m2*g*y2m)
LM_dp = LagrangesMethod(L_dp_m, [q1m, q2m])
eqs_dp_m = LM_dp.form_lagranges_equations()
for eq in eqs_dp_m:
    print(sp.Eq(eq, 0))

In [ ]:
# H) Discusión + I) linealización conceptual
print('Las ecuaciones exactas son no lineales y acopladas; no existe solución cerrada general simple.')
print('Para oscilaciones pequeñas, se puede linealizar (sin(theta)~theta, cos(theta)~1).')
print('Alternativa práctica: integrar numéricamente (por ejemplo con solve_ivp).')

### 6.2 Rodadura sin deslizamiento en plano inclinado (esfera maciza)

**A)** Esfera de radio \(R\) y masa \(m\), inclinación \(\alpha\).  
**B)** Coordenada \(s(t)\): desplazamiento del centro a lo largo del plano (positiva hacia abajo).

Con la restricción ideal \(s=R\varphi\), se reduce a 1 grado de libertad.

In [ ]:
# C,D,E
m, R, alpha, g = sp.symbols('m R alpha g', positive=True, real=True)
s = sp.Function('s')(t)
I_sphere = sp.Rational(2,5)*m*R**2  # momento de inercia de una esfera maciza respecto a su centro
sd = sp.diff(s,t)

T_roll = sp.Rational(1,2)*m*sd**2 + sp.Rational(1,2)*I_sphere*(sd/R)**2
V_roll = -m*g*s*sp.sin(alpha)
L_roll = sp.simplify(T_roll - V_roll)

print(sp.Eq(sp.Symbol('L'), L_roll))

In [ ]:
# F) Manual
eq_roll = euler_lagrange_manual(L_roll, [s], t)[0]
print(eq_roll)

a_roll = sp.solve(sp.Eq(eq_roll.lhs, 0), sp.diff(s,t,2))[0]
print(sp.Eq(sp.diff(s,t,2), sp.simplify(a_roll)))

In [ ]:
# G) mechanics
s_m = dynamicsymbols('s_m')
L_roll_m = sp.Rational(1,2)*(m + I_sphere/R**2)*s_m.diff(t_mechanics)**2 + m*g*sp.sin(alpha)*s_m
LM_roll = LagrangesMethod(L_roll_m, [s_m])
eq_roll_m = sp.simplify(LM_roll.form_lagranges_equations()[0])
print(sp.Eq(eq_roll_m, 0))

In [ ]:
# H, I) comparación con resultado conocido
a_teorica_expr = sp.simplify(g*sp.sin(alpha)/(1 + I_sphere/(m*R**2)))
print(sp.Eq(sp.Symbol('a_teorica'), a_teorica_expr))
print('Para esfera maciza I=(2/5)mR^2 -> a = (5/7) g sin(alpha).')

## 7. Problemas 7 y 8

---

### 7.1 Bead on a rotating hoop

**A)** Una cuenta de masa \(m\) se mueve sin fricción en un aro de radio \(a\),
que rota con velocidad angular constante \(\Omega\) alrededor del eje vertical.

**B)** Coordenada \(\theta(t)\): ángulo medido desde la posición inferior del aro.

In [ ]:
# C,D,E
m, a, Omega, g = sp.symbols('m a Omega g', positive=True, real=True)
theta = sp.Function('theta')(t)
thetad = sp.diff(theta, t)

T_b = sp.Rational(1,2)*m*a**2*(thetad**2 + Omega**2*sp.sin(theta)**2)
V_b = m*g*a*(1-sp.cos(theta))
L_b = sp.simplify(T_b - V_b)

print(sp.Eq(sp.Symbol('L'), L_b))

In [ ]:
# F) Manual
eq_b = euler_lagrange_manual(L_b, [theta], t)[0]
print(eq_b)

In [ ]:
# G) mechanics
th_b = dynamicsymbols('th_b')
L_b_m = sp.Rational(1,2)*m*a**2*(th_b.diff(t_mechanics)**2 + Omega**2*sp.sin(th_b)**2) - m*g*a*(1-sp.cos(th_b))
LM_b = LagrangesMethod(L_b_m, [th_b])
eq_b_m = sp.simplify(LM_b.form_lagranges_equations()[0])
print(sp.Eq(eq_b_m, 0))

In [ ]:
# H, I) Estabilidad mediante potencial efectivo
U_eff = m*g*a*(1-sp.cos(theta)) - sp.Rational(1,2)*m*a**2*Omega**2*sp.sin(theta)**2
dU = sp.simplify(sp.diff(U_eff, theta))
print(sp.Eq(sp.Symbol('U_eff'), U_eff))
print(sp.Eq(sp.Symbol('dU/dtheta'), dU))
print('Equilibrios: sin(theta)*(g - a*Omega^2*cos(theta)) = 0.')

th_vals = np.linspace(-np.pi, np.pi, 600)
eta = 0.7  # parámetro adimensional para ilustrar una razón fija g/(a*Omega^2)
U_vals = (1-np.cos(th_vals)) - eta*np.sin(th_vals)**2
plt.figure(figsize=(6,3))
plt.plot(th_vals, U_vals)
plt.xlabel(r'$\theta$')
plt.ylabel(r'$U_{eff}$ (adimensional)')
plt.title('Bead on hoop: forma cualitativa de $U_{eff}$')
plt.grid(True, alpha=0.3)
plt.show()

### 7.2 Sistema con disipación vía función de Rayleigh (oscilador amortiguado)

**A)** Masa–resorte con amortiguamiento lineal \(c\dot x\).  
**B)** Coordenada \(x(t)\).

Usamos:
\[
R=\frac{1}{2}c\dot x^2,
\qquad
\frac{d}{dt}\left(\frac{\partial L}{\partial \dot x}\right)-\frac{\partial L}{\partial x}+\frac{\partial R}{\partial \dot x}=0.
\]

In [ ]:
# C,D,E,F
m, k, c = sp.symbols('m k c', positive=True, real=True)
x = sp.Function('x')(t)
xd = sp.diff(x,t)

T_d = sp.Rational(1,2)*m*xd**2
V_d = sp.Rational(1,2)*k*x**2
L_d = T_d - V_d
R_d = sp.Rational(1,2)*c*xd**2

eq_d = euler_lagrange_con_rayleigh(L_d, R_d, [x], t)[0]
print(eq_d)
print(sp.Eq(sp.simplify(eq_d.lhs/m), 0))

In [ ]:
# G) Uso de dynamicsymbols (enfoque mechanics para variables dinámicas)
x_d = dynamicsymbols('x_d')
L_dm = sp.Rational(1,2)*m*x_d.diff(t_mechanics)**2 - sp.Rational(1,2)*k*x_d**2
R_dm = sp.Rational(1,2)*c*x_d.diff(t_mechanics)**2

eq_dm = sp.Eq(
    sp.diff(sp.diff(L_dm, x_d.diff(t_mechanics)), t_mechanics)
    - sp.diff(L_dm, x_d)
    + sp.diff(R_dm, x_d.diff(t_mechanics)),
    0
)
print(eq_dm)

In [ ]:
# H, I) discusión física
print('La energía mecánica E=T+V ya no se conserva para c>0.')
print('dsolve puede resolver algunos casos lineales; en no lineales suele requerirse integración numérica.')
sol_d = sp.dsolve(sp.Eq(m*sp.diff(x,t,2) + c*sp.diff(x,t) + k*x, 0))
print(sol_d)

## 8. Comparativa: Manual vs `sympy.physics.mechanics`

### Enfoque manual (`diff`, `simplify`, `trigsimp`)
**Ventajas**
- Control fino de cada paso.
- Transparencia didáctica total.

**Desventajas**
- Más propenso a errores algebraicos en sistemas grandes.
- Requiere más código repetitivo.

### Enfoque `sympy.physics.mechanics`
**Ventajas**
- Estructura limpia para sistemas multicuerpo.
- `LagrangesMethod` automatiza ensamblaje de ecuaciones.
- `ReferenceFrame` y `Point` ayudan a cinemática consistente.

**Desventajas**
- Curva de aprendizaje de la API.
- A veces las expresiones devueltas requieren simplificación adicional.

## 9. Buenas prácticas y errores comunes

### Buenas prácticas
1. Elegir coordenadas que minimicen restricciones explícitas.
2. Declarar supuestos (`positive=True`, `real=True`) para simplificar resultados.
3. Verificar límites físicos (ángulo pequeño, masas iguales, etc.).
4. Separar derivación simbólica y análisis numérico.

### Errores comunes
- Confundir `q` (símbolo) con `q(t)` (función).
- Signos incorrectos en energía potencial gravitatoria.
- Usar `qdot` como símbolo independiente sin consistencia de derivadas.
- Forzar simplificaciones muy costosas en expresiones gigantes (p. ej. péndulo doble).

## 10. Ejercicios para estudiantes

1. **Péndulo con soporte móvil**: el punto de suspensión se mueve como \(x_s=A\cos\omega t\). Deriva la EOM para \(\theta\).
2. **Atwood con polea masiva**: incluye inercia \(I_p\) de la polea y obtén la aceleración.
3. **Oscilador anharmónico**: \(V(x)=\frac{1}{2}kx^2+\lambda x^4\). Deriva E-L y linealiza cerca de \(x=0\).
4. **Potencial central kepleriano**: \(V(r)=-\mu m/r\). Obtén \(U_{\text{eff}}\) y condición de órbita circular.
5. **Péndulo doble simétrico**: linealiza para ángulos pequeños y encuentra frecuencias normales.
6. **Cilindro sólido en plano inclinado**: repite el problema de rodadura y compara con esfera maciza.
7. **Bead on hoop**: determina estabilidad de \(\theta=0\) según \(\Omega\).
8. **Amortiguado forzado**: agrega fuerza externa \(F_0\cos(\omega t)\) al oscilador con Rayleigh.
9. **Coordenadas cíclicas**: propone un sistema mecánico con al menos una coordenada cíclica e identifica su integral de movimiento.
10. **Validación dimensional**: toma dos ecuaciones obtenidas en la lección y verifica consistencia de unidades término a término.

## 11. Referencias

1. H. Goldstein, C. Poole, J. Safko, *Classical Mechanics*, 3.ª ed., Addison-Wesley, 2001.
2. L. D. Landau, E. M. Lifshitz, *Mechanics* (Course of Theoretical Physics, Vol. 1), 3.ª ed., Butterworth-Heinemann, 1976.
3. V. I. Arnold, *Mathematical Methods of Classical Mechanics*, 2.ª ed., Springer, 1989.
4. J. V. José, E. J. Saletan, *Classical Dynamics: A Contemporary Approach*, Cambridge University Press, 1998.
5. Documentación oficial de SymPy:
   - `sympy` (núcleo simbólico)
   - `sympy.physics.mechanics` (dinámica analítica)
6. Notas de curso de Mecánica Clásica y Mecánica Analítica (material institucional).

---

**Cierre:** cuando `dsolve()` no produce forma cerrada útil, el flujo recomendado es: 
linealizar (si aplica), estudiar estabilidad cualitativa y luego pasar a integración numérica.